## 1. Environment Preparation

Install Unsloth and updated HuggingFace libraries for Mistral support.

In [ ]:
# Install core packages from PyPI (much faster than git installs)
!pip install -q unsloth transformers trl peft accelerate datasets bitsandbytes

# Verify installations
import unsloth
import transformers
import trl
print(f"✓ Unsloth: {unsloth.__version__}")
print(f"✓ Transformers: {transformers.__version__}")
print(f"✓ TRL: {trl.__version__}")
print("Environment ready!")

## 2. Load Dataset & Format for Instruction Tuning

Load the Augmentoolkit-generated Marcus Aurelius dataset (first-person Stoic responses from Meditations).


In [ ]:
from datasets import load_dataset, concatenate_datasets
import glob

# Load ONLY clean data types (openended, vague, hallucination)
# EXCLUDE: followup (follow-up questions) and negative (error-correction that causes question repetition)
base_path = "/home/spark/projects/augmentoolkit/outputs/marcus_aurelius_dataset"

# Find all directories by type
openended_dirs = glob.glob(f"{base_path}/factual_sft_stoics_openended_*")
vague_dirs = glob.glob(f"{base_path}/factual_sft_stoics_vague_*")
hallucination_dirs = glob.glob(f"{base_path}/factual_sft_stoics_hallucination_*")

# Combine all allowed directory paths
allowed_dirs = openended_dirs + vague_dirs + hallucination_dirs

print(f"Found {len(openended_dirs)} openended variations")
print(f"Found {len(vague_dirs)} vague variations")
print(f"Found {len(hallucination_dirs)} hallucination variations")
print(f"Total: {len(allowed_dirs)} data directories")

# Load all JSONL files from allowed directories
datasets = []
total_examples = 0

for dir_path in sorted(allowed_dirs):
    jsonl_files = glob.glob(f"{dir_path}/*.jsonl")
    for file_path in jsonl_files:
        ds = load_dataset("json", data_files=file_path, split="train")
        datasets.append(ds)
        total_examples += len(ds)
        print(f"  Loaded {len(ds)} examples from {dir_path.split('/')[-1]}/{file_path.split('/')[-1]}")

dataset = concatenate_datasets(datasets)

print(f"\n✓ Total dataset loaded: {len(dataset)} examples")
print(f"✓ Columns: {dataset.column_names}")
print(f"\n--- First Example (first 800 chars) ---")
import json
print(json.dumps(dataset[0], indent=2)[:800])

# Shuffle dataset for better training
dataset = dataset.shuffle(seed=42)
print(f"\n✓ Dataset shuffled and ready for training")


## 3. Load Model & Tokenizer with Unsloth

Load Mistral-7B-Instruct model in FULL 16-bit precision (no quantization).

In [ ]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/mistral-7b-instruct-v0.3"
max_seq_length = 2048

# Load model in FULL 16-bit precision (no quantization)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name,
    max_seq_length=max_seq_length,
    dtype=torch.float16,  # Full 16-bit precision
    load_in_4bit=False,   # NO quantization
    device_map={"": 0}    # Force all on GPU 0
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"✓ Model loaded: {model_name}")
print(f"✓ Precision: FULL 16-bit (fp16)")
print(f"✓ Tokenizer configured")
print(f"✓ Max sequence length: {max_seq_length}")

In [ ]:
# Format ShareGPT dataset for Mistral chat template
def format_instruct(example):
    # Augmentoolkit uses ShareGPT format: "conversations" field with from/value
    # Convert to Mistral format (role/content)
    messages = []
    for turn in example["conversations"]:
        # ShareGPT uses "from": "system"/"human"/"gpt"
        # Mistral uses "role": "system"/"user"/"assistant"
        if turn["from"] == "system":
            messages.append({"role": "system", "content": turn["value"]})
        elif turn["from"] == "human":
            messages.append({"role": "user", "content": turn["value"]})
        elif turn["from"] == "gpt":
            messages.append({"role": "assistant", "content": turn["value"]})
    
    text = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=False
    )
    return {"text": text}

# Format and remove old columns
dataset = dataset.map(format_instruct, remove_columns=dataset.column_names)

print(f"✓ Dataset formatted: {len(dataset)} examples")
print(f"\n--- Sample formatted text (first 500 chars) ---")
print(dataset[0]['text'][:500])


## 4. Add LoRA Adapters

Configure LoRA for efficient fine-tuning with attention and MLP projection layers.


In [ ]:
from peft import LoraConfig

# Add LoRA adapters using Unsloth's method
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=max_seq_length
)

print("✓ LoRA adapters added successfully")
print(f"✓ Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


## 5. Trainer Setup & Training

**Augmentoolkit Dataset:**
- First-person Stoic responses generated from Marcus Aurelius' Meditations
- ~735 high-quality examples across 40 subdirectories (5 SFT types × 8 variations)
- Designed to teach the model to speak AS a Stoic, not about Stoicism

**Training Configuration:**
- Cosine LR decay for smooth convergence
- 3-5 epochs recommended for this dataset size
- System prompts already embedded in Augmentoolkit data


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Dynamic training configuration based on actual dataset size
batch_size = 2
grad_accum = 4
effective_batch_size = batch_size * grad_accum

# Calculate steps for 3 epochs (safer to avoid overfitting)
steps_per_epoch = len(dataset) // effective_batch_size
target_epochs = 3
max_steps = steps_per_epoch * target_epochs

# Set warmup to ~10% of total steps, save every epoch
warmup_steps = max(1, max_steps // 10)
save_steps = steps_per_epoch

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        warmup_steps=warmup_steps,
        max_steps=max_steps,
        learning_rate=2e-4,
        fp16=True,   # Model loaded in fp16, must train in fp16
        bf16=False,  # Disable bf16 since we're using fp16
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="./stoic_instruct_augmentoolkit_finetune",
        report_to="none",
        save_strategy="steps",
        save_steps=save_steps,
    )
)

print("✓ Trainer configured for Augmentoolkit dataset")
print(f"✓ Dataset size: {len(dataset)} conversations")
print(f"✓ Effective batch size: {effective_batch_size} (batch={batch_size} × grad_accum={grad_accum})")
print(f"✓ Steps per epoch: {steps_per_epoch}")
print(f"✓ Total epochs: {target_epochs}")
print(f"✓ Total steps: {max_steps}")
print(f"✓ Warmup steps: {warmup_steps}")
print(f"✓ Save every: {save_steps} steps (every epoch)")


In [ ]:
# Start training
trainer.train()

## 7. Save Model & Inference

Save the fine-tuned model and test inference with a Stoic question.

In [ ]:
# Save merged model
model.save_pretrained_merged("./stoic_instruct_augmentoolkit_model", tokenizer, save_method="merged_16bit")

print("Model saved to ./stoic_instruct_augmentoolkit_model")


In [ ]:
# Prepare model for inference
FastLanguageModel.for_inference(model)

# Test inference with a Stoic question
test_prompt = "What troubles me today is the judgment of others. How should I view this?"

inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": test_prompt}],
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    temperature=0.7, 
    top_p=0.9,
    repetition_penalty=1.1
)
response = tokenizer.decode(outputs[0], skip_special_tokens=False)

print("\n=== RAW FULL OUTPUT (with tags) ===")
print(response)
print("\n=== PARSED OUTPUT ===")
print(f"User: {test_prompt}")
print(f"\nAssistant: {response.split('[/INST]')[-1].strip()}")


## Notes

### Dataset Quality
This notebook uses Augmentoolkit-generated data from Marcus Aurelius' Meditations. The pipeline enforced first-person responses through configuration:
- `shared_instruction`: "You ARE a Stoic philosopher - not explaining Stoicism, but LIVING it."
- Source text: Only Meditations (pure first-person journal)
- All prompts rewritten to enforce "I am a Stoic philosopher..." voice

### Next Steps
- For Epictetus dataset: Run Augmentoolkit with Enchiridion/Discourses
- Merge datasets: Combine Marcus + Epictetus for broader Stoic knowledge
- Evaluate first-person quality: Test if model says "I practice..." vs "Stoics believe..."

### Dataset Location
Augmentoolkit output: `/home/spark/projects/augmentoolkit/outputs/marcus_aurelius_dataset/sft_run/axolotl_rag_conversations_stoics.jsonl`


## 8. Convert to GGUF for Ollama

Convert the fine-tuned model to GGUF format for use with Ollama. Supports both full-precision (fp32) and quantized formats (q4, q8, etc.) for different size/quality trade-offs.

**Quantization Options:**
- `None`: Full precision (fp32) - maximum quality, largest size
- `"q4"`: 4-bit (Q4_0) - highest compression, fastest inference
- `"q8"`: 8-bit (Q8_0) - best balance of size/speed/quality ⭐ **Recommended**
- `"q4_k"`: 4-bit K-quant (Q4_K_M) - better quality than q4
- `"q5_k"`: 5-bit K-quant (Q5_K_M) - excellent quality, moderate size
- `"q6_k"`: 6-bit K-quant (Q6_K) - near-lossless quality

In [ ]:
from pathlib import Path
import subprocess
import sys

# ============================================================================
# GGUF CONVERSION CONFIGURATION
# ============================================================================
LLAMA_CPP_PATH = Path("/home/spark/resources/llama.cpp")
SOURCE_MODEL_DIR = Path("./stoic_instruct_augmentoolkit_model")

# Set to "q8" for best balance, "q4_k" for production, or None for full precision
QUANTIZATION_TYPE = "q8"  # Options: None, "q4", "q8", "q4_k", "q5_k", "q6_k", "fp16"

# ============================================================================

# Create output directory based on quantization type
quant_suffix = "fp32" if QUANTIZATION_TYPE is None else QUANTIZATION_TYPE
OUTPUT_DIR = Path(f"./stoic_ollama_{quant_suffix}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"🔧 GGUF Conversion for Ollama")
print(f"   Source model: {SOURCE_MODEL_DIR}")
print(f"   Quantization: {QUANTIZATION_TYPE or 'Full Precision (fp32)'}")
print(f"   Output: {OUTPUT_DIR}")

# Verify llama.cpp exists
if not LLAMA_CPP_PATH.exists():
    raise FileNotFoundError(
        f"❌ llama.cpp not found at {LLAMA_CPP_PATH}\n"
        f"   This DGX uses a shared resources folder.\n"
        f"   Please clone it: git clone https://github.com/ggerganov/llama.cpp {LLAMA_CPP_PATH}\n"
        f"   Then build it: cd {LLAMA_CPP_PATH} && cmake -B build -DLLAMA_CURL=OFF && cmake --build build -j$(nproc)"
    )

print(f"✓ llama.cpp found at {LLAMA_CPP_PATH}")

In [ ]:
if QUANTIZATION_TYPE is None:
    # ========================================================================
    # Full Precision (fp32) - No Quantization
    # ========================================================================
    print("\n[Step 1/1] Converting to full-precision GGUF (fp32)...")
    
    FINAL_GGUF = OUTPUT_DIR / "stoic-fp32.gguf"
    
    subprocess.run([
        sys.executable,
        str(LLAMA_CPP_PATH / "convert_hf_to_gguf.py"),
        str(SOURCE_MODEL_DIR),
        "--outfile",
        str(FINAL_GGUF),
        "--outtype",
        "f32",  # Full 32-bit precision
    ], check=True)
    
    print(f"   ✅ Full-precision GGUF: {FINAL_GGUF.name}")
    
else:
    # ========================================================================
    # Quantized Conversion (2-step process)
    # ========================================================================
    print("\n[Step 1/2] Converting to fp16 GGUF (pre-quantization)...")
    
    TEMP_GGUF = OUTPUT_DIR / "temp-fp16.gguf"
    
    subprocess.run([
        sys.executable,
        str(LLAMA_CPP_PATH / "convert_hf_to_gguf.py"),
        str(SOURCE_MODEL_DIR),
        "--outfile",
        str(TEMP_GGUF),
        "--outtype",
        "f16",  # 16-bit precision (required for quantization)
    ], check=True)
    
    print(f"   ✅ fp16 GGUF created: {TEMP_GGUF.name}")
    
    # Step 2: Quantize the fp16 GGUF
    print(f"\n[Step 2/2] Quantizing to {QUANTIZATION_TYPE}...")
    
    FINAL_GGUF = OUTPUT_DIR / f"stoic-{QUANTIZATION_TYPE}.gguf"
    
    # Map friendly names to llama.cpp quantization types
    quant_map = {
        "q4": "Q4_0",
        "q8": "Q8_0",
        "q4_k": "Q4_K_M",
        "q5_k": "Q5_K_M",
        "q6_k": "Q6_K",
        "fp16": "F16",
    }
    llama_quant_type = quant_map.get(QUANTIZATION_TYPE, QUANTIZATION_TYPE.upper())
    
    # Use llama-quantize binary from CMake build
    quantize_binary = LLAMA_CPP_PATH / "build" / "bin" / "llama-quantize"
    
    if not quantize_binary.exists():
        raise FileNotFoundError(
            f"❌ llama-quantize binary not found at {quantize_binary}\n"
            f"   Please build llama.cpp:\n"
            f"   cd {LLAMA_CPP_PATH} && cmake -B build -DLLAMA_CURL=OFF && cmake --build build --config Release -j$(nproc)"
        )
    
    subprocess.run([
        str(quantize_binary),
        str(TEMP_GGUF),
        str(FINAL_GGUF),
        llama_quant_type,
    ], check=True)
    
    # Clean up temp file
    TEMP_GGUF.unlink()
    print(f"   ✅ Quantized GGUF: {FINAL_GGUF.name} ({llama_quant_type})")

print(f"\n✅ GGUF conversion complete: {FINAL_GGUF}")

In [ ]:
# Create Modelfile for Ollama
print("\n📝 Creating Modelfile for Open WebUI...")

MODELFILE_PATH = OUTPUT_DIR / "Modelfile"

# Minimal Modelfile for Open WebUI (it manages system prompts and parameters via UI)
# Only include the GGUF reference and chat template
modelfile_content = f"""FROM ./{FINAL_GGUF.name}

TEMPLATE \"\"\"[INST] {{{{ .Prompt }}}} [/INST]\"\"\"
"""

MODELFILE_PATH.write_text(modelfile_content, encoding="utf-8")

print(f"✅ Modelfile created: {MODELFILE_PATH}")
print(f"\n🚀 Ready for Open WebUI!")
print(f"\nTo import into Ollama:")
print(f"   cd {OUTPUT_DIR}")
print(f"   ollama create stoic -f Modelfile")
print(f"\nThen configure in Open WebUI:")
print(f"   - Select 'stoic' as base model")
print(f"   - Add custom system prompt")
print(f"   - Set temperature, top_p, etc. via UI sliders")